<a href="https://colab.research.google.com/github/chiraglamba27/UCS420_Chirag_Lamba_1024170138_Assignments/blob/main/Assign_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
import math
import nltk
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from string import punctuation
from nltk import word_tokenize, pos_tag, bigrams
from nltk.corpus import wordnet
from nltk.tokenize import RegexpTokenizer
from nltk.stem import PorterStemmer, LancasterStemmer, RegexpStemmer, WordNetLemmatizer

for package in ["punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4", "averaged_perceptron_tagger", "averaged_perceptron_tagger_eng"]:
    nltk.download(package, quiet=True)

def pick(res, *names):
    for name in names:
        if name in res:
            return res[name]
    return []

def size(x):
    return x if isinstance(x, int) else len(x)

results = [process_ticket(t) for t in tickets]
tok_lists = [list(pick(r, "tokens", "all_tokens")) for r in results]
punct_lists = [list(pick(r, "punct", "punctuation", "punct_tokens", "punctuation_tokens")) for r in results]
stop_lists = [list(pick(r, "stopwords", "stop_words", "stops")) for r in results]
final_lists = [list(pick(r, "final", "final_tokens", "filtered")) for r in results]

rows = []
for i, r in enumerate(results):
    rows.append({
        "ticket": i + 1,
        "sentences": size(pick(r, "sentences", "sents", "num_sentences")),
        "tokens": len(tok_lists[i]),
        "punctuation": len(punct_lists[i]),
        "stopwords": len(stop_lists[i]),
        "unique_tokens": len(set(tok_lists[i])),
        "final_tokens": len(final_lists[i]),
    })

table = pd.DataFrame(rows)
total = table.drop(columns="ticket").sum()
total["ticket"] = "Total"
table = pd.concat([table, total.to_frame().T], ignore_index=True)
print(table.to_string(index=False))

pick_no = 0
print("\nTicket", pick_no + 1)
print(tickets[pick_no])
print("Tokens:", tok_lists[pick_no])
print("Punctuation:", punct_lists[pick_no])
print("Stopwords:", stop_lists[pick_no])
print("Final tokens:", final_lists[pick_no])

long_words = r"\b[A-Za-z]{6,}\b"
numbers = r"\b\d+(?:\.\d+)*\b"
capitalized = r"\b[A-Z][a-z]*\b"
vowel_words = r"\b[AEIOUaeiou][A-Za-z]*\b"
email = r"[\w.+-]+@[\w-]+(?:\.[\w-]+)+"
url = r"(?:https?://|www\.)[^\s<>\"']*[^\s<>\"'.,;:!?)]"
phone = r"(?:\+\d{1,3}[\s.-]?)?(?:\(\d{3}\)\s?|\d{3}[\s.-])\d{3}[\s.-]\d{4}\b|\+\d{1,3}[\s-]?\d{5}[\s-]?\d{5}\b"

def extract(text):
    return {
        "long words": re.findall(long_words, text),
        "numbers": re.findall(numbers, text),
        "capitalized": re.findall(capitalized, text),
        "vowel words": re.findall(vowel_words, text),
    }

def mask(text):
    text = re.sub(email, "<EMAIL>", text)
    text = re.sub(url, "<URL>", text)
    text = re.sub(phone, "<PHONE>", text)
    return text

for line in TEST_LINES:
    print("LINE:", line)
    for name, found in extract(line).items():
        print(" ", name, found)
    print("  masked:", mask(line))
    print()

for i, t in enumerate(tickets):
    print("Ticket", i + 1)
    for name, found in extract(t).items():
        print(" ", name, found)
    print("  masked:", mask(t))
    print()

token_pattern = r"""(?x)
    [\w.+-]+@[\w-]+(?:\.[\w-]+)+
    | (?:https?://|www\.)[^\s<>"']*[^\s<>"'.,;:!?)]
    | (?:\+\d{1,3}[\s.-]?)?(?:\(\d{3}\)\s?|\d{3}[\s.-])\d{3}[\s.-]\d{4}\b
    | \+\d{1,3}[\s-]?\d{5}[\s-]?\d{5}\b
    | \d+(?:\.\d+)+
    | \w+(?:[-']\w+)*
    | [^\w\s]
"""

tokenizer = RegexpTokenizer(token_pattern)

def my_tokenize(text):
    return tokenizer.tokenize(text)

for line in TEST_LINES:
    print("LINE:", line)
    print("  mine:", my_tokenize(line))
    print("  nltk:", word_tokenize(line))
    print()

for i, t in enumerate(tickets):
    print("Ticket", i + 1)
    print("  mine:", my_tokenize(t))
    print("  nltk:", word_tokenize(t))
    print()

diffs = []
for text in list(TEST_LINES) + list(tickets):
    mine = my_tokenize(text)
    theirs = word_tokenize(text)
    only_mine = [w for w in mine if w not in theirs]
    only_theirs = [w for w in theirs if w not in mine]
    if only_mine or only_theirs:
        diffs.append({"text": text[:50], "only_my_tokenize": only_mine, "only_word_tokenize": only_theirs})

diff_table = pd.DataFrame(diffs)
print(diff_table.to_string(index=False))

final_all = [w for words in final_lists for w in words]
unique_final = sorted(set(final_all))

suffixes = ["ing", "ed", "ly", "er", "es", "s"]
suffix_words = {s: [w for w in unique_final if w.endswith(s) and len(w) > len(s) + 2] for s in suffixes}
for s in suffixes:
    print(s, len(suffix_words[s]), suffix_words[s][:6])

best_suffix = max(suffixes, key=lambda s: len(suffix_words[s]))
print("\nChosen suffix:", best_suffix)
print("Dataset words that use it:", suffix_words[best_suffix][:6])

porter = PorterStemmer()
lancaster = LancasterStemmer()
regexp = RegexpStemmer(best_suffix + "$", min=4)
lemmatizer = WordNetLemmatizer()

def wn_tag(word):
    tag = pos_tag([word])[0][1]
    if tag.startswith("J"):
        return wordnet.ADJ
    if tag.startswith("V"):
        return wordnet.VERB
    if tag.startswith("R"):
        return wordnet.ADV
    return wordnet.NOUN

stem_table = pd.DataFrame({
    "word": unique_final,
    "porter": [porter.stem(w) for w in unique_final],
    "lancaster": [lancaster.stem(w) for w in unique_final],
    "regexp": [regexp.stem(w) for w in unique_final],
    "lemma": [lemmatizer.lemmatize(w, wn_tag(w)) for w in unique_final],
})

changed = stem_table[stem_table.iloc[:, 1:].nunique(axis=1) > 1]
print(changed.to_string(index=False))

s1 = [w for words in tok_lists for w in words]
s2 = [w for w in s1 if not all(c in punctuation for c in w)]
s3 = final_all
s4 = [porter.stem(w) for w in s3]

def stats(tokens):
    return len(tokens), len(set(tokens)), round(len(set(tokens)) / len(tokens), 4)

summary = pd.DataFrame(
    [stats(s) for s in (s1, s2, s3, s4)],
    index=["S1", "S2", "S3", "S4"],
    columns=["total_tokens", "unique_tokens", "TTR"],
)
print(summary)

print("\nTop 10 S1:", Counter(s1).most_common(10))
print("Top 10 S3:", Counter(s3).most_common(10))

freqs = sorted(Counter(s3).values(), reverse=True)
log_rank = [math.log(r) for r in range(1, len(freqs) + 1)]
log_freq = [math.log(f) for f in freqs]

plt.figure(figsize=(7, 5))
plt.plot(log_rank, log_freq, marker="o")
plt.xlabel("log(rank)")
plt.ylabel("log(frequency)")
plt.title("Rank vs frequency for S3")
plt.grid(True)
plt.show()

def top_bigrams(lists, n=5):
    counts = Counter()
    for words in lists:
        counts.update(bigrams(words))
    return counts.most_common(n)

print("\nTop 5 bigrams S1:", top_bigrams(tok_lists))
print("Top 5 bigrams S3:", top_bigrams(final_lists))

ttr = summary["TTR"]
print("(a) TTR for S1, S2, S3, S4:", list(ttr))
print("Punctuation and stopwords repeat a lot, so removing them cuts the total much more than the unique count and TTR goes up.")
print("Stemming then merges forms like plural and verb endings into one stem, so the unique count drops and TTR comes down a little from S3 to S4.")

print("\n(b)")
for line in TEST_LINES:
    mine = my_tokenize(line)
    theirs = word_tokenize(line)
    if mine != theirs:
        print("Line:", line)
        print("my_tokenize:", mine)
        print("word_tokenize:", theirs)
        print("The difference is in how each one splits it. My single pattern keeps emails, URLs, phone numbers, hyphenated words and contractions whole, while word_tokenize follows Treebank rules and breaks them at punctuation or clitics.")
        break

print("\n(c)")
bad = stem_table[stem_table["word"].str.isalpha() & (stem_table["word"] != stem_table["porter"]) & ~stem_table["porter"].apply(lambda s: bool(wordnet.synsets(s)))]
for _, row in bad.head(3).iterrows():
    print(row["word"], "->", row["porter"], "(not a real word)")
print("Stems like these are fine for matching but are unreadable and can merge unrelated words.")

print("\n(d)")
good = stem_table[stem_table["word"].str.isalpha() & (stem_table["lemma"] != stem_table["word"]) & (stem_table["lemma"] != stem_table["porter"])]
for _, row in good.head(3).iterrows():
    print(row["word"], "-> lemma:", row["lemma"], "| porter:", row["porter"])
print("The lemmatizer uses the part of speech and returns a real dictionary word, so the meaning is kept, while the stemmer just chops the ending.")